In [8]:
from pathlib import Path
from PIL import Image
import shutil
import json

# =========================
# 1. PATHS
# =========================

USABLE_REF_DIR = Path(
    "/Users/williamtsai/Desktop/nail_unusable_classifier/dataset/eyeimage5_annotated"
)

MAIN_MIXED_DIR = Path(
    "/Users/williamtsai/Desktop/nail_unusable_classifier/dataset/eyeimage5_renew"
)

OUTPUT_ROOT = Path(
    "/Users/williamtsai/Desktop/nail_unusable_classifier/split_from_main"
)

USABLE_OUT_DIR = OUTPUT_ROOT / "usable_png"
EXCLUDED_OUT_DIR = OUTPUT_ROOT / "excluded_png"

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("USABLE_REF_DIR exists:", USABLE_REF_DIR.exists())
print("MAIN_MIXED_DIR exists:", MAIN_MIXED_DIR.exists())

if not USABLE_REF_DIR.exists():
    raise FileNotFoundError(f"Usable reference folder not found: {USABLE_REF_DIR}")

if not MAIN_MIXED_DIR.exists():
    raise FileNotFoundError(f"Main mixed folder not found: {MAIN_MIXED_DIR}")

# =========================
# 2. CLEAN OUTPUT FOLDERS
# =========================

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

USABLE_OUT_DIR.mkdir(parents=True, exist_ok=True)
EXCLUDED_OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 3. FILTER JUNK PATHS
# =========================

def is_valid_path(p):
    parts_lower = [part.lower() for part in p.parts]

    if "__macosx" in parts_lower:
        return False

    if p.name.startswith("._"):
        return False

    if any(part in ["thumb", "thumbs", "thumbnail", "thumbnails"] for part in parts_lower):
        return False

    return True

# =========================
# 4. GET USABLE STEMS FROM ANNOTATED FOLDER
# =========================

usable_ref_images = [
    p for p in USABLE_REF_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in image_exts
    and is_valid_path(p)
]

usable_stems = {p.stem for p in usable_ref_images}

print("Usable reference images:", len(usable_ref_images))
print("Unique usable stems:", len(usable_stems))

# =========================
# 5. GET MAIN MIXED IMAGES + JSONS
# =========================

main_images = [
    p for p in MAIN_MIXED_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in image_exts
    and is_valid_path(p)
]

main_jsons = [
    p for p in MAIN_MIXED_DIR.rglob("*.json")
    if p.is_file()
    and is_valid_path(p)
]

json_by_stem = {p.stem: p for p in main_jsons}

print("Main mixed images:", len(main_images))
print("Main mixed JSONs:", len(main_jsons))

# =========================
# 6. COPY AS PNG + COPY JSON
# =========================

usable_count = 0
excluded_count = 0
json_copied = 0
failed = []

for src_img in main_images:
    stem = src_img.stem

    if stem in usable_stems:
        dst_dir = USABLE_OUT_DIR
        usable_count += 1
    else:
        dst_dir = EXCLUDED_OUT_DIR
        excluded_count += 1

    dst_png_name = stem + ".png"
    dst_png_path = dst_dir / dst_png_name

    # save image as PNG
    try:
        with Image.open(src_img) as im:
            if im.mode not in ("RGB", "RGBA"):
                im = im.convert("RGB")
            im.save(dst_png_path, format="PNG")
    except Exception as e:
        failed.append((src_img.name, str(e)))
        continue

    # copy matching JSON if available, update imagePath
    src_json = json_by_stem.get(stem)

    if src_json is not None:
        dst_json_path = dst_dir / f"{stem}.json"

        try:
            with open(src_json, "r", encoding="utf-8") as f:
                data = json.load(f)

            if "imagePath" in data:
                data["imagePath"] = dst_png_name

            with open(dst_json_path, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)

            json_copied += 1

        except Exception:
            shutil.copy2(src_json, dst_json_path)
            json_copied += 1

print("\nDone.")
print("Output root:", OUTPUT_ROOT)
print("Usable PNGs:", usable_count)
print("Excluded PNGs:", excluded_count)
print("JSONs copied:", json_copied)
print("Failed images:", len(failed))

if failed:
    print("\nFirst failed images:")
    for name, err in failed[:20]:
        print(name, "->", err)

USABLE_REF_DIR exists: True
MAIN_MIXED_DIR exists: True
Usable reference images: 903
Unique usable stems: 903
Main mixed images: 964
Main mixed JSONs: 0

Done.
Output root: /Users/williamtsai/Desktop/nail_unusable_classifier/split_from_main
Usable PNGs: 862
Excluded PNGs: 102
JSONs copied: 0
Failed images: 0


In [9]:
from pathlib import Path

OUTPUT_ROOT = Path("/Users/williamtsai/Desktop/nail_unusable_classifier/split_from_main")

for folder_name in ["usable_png", "excluded_png"]:
    folder = OUTPUT_ROOT / folder_name

    pngs = list(folder.glob("*.png"))
    jsons = list(folder.glob("*.json"))
    jpgs = list(folder.glob("*.jpg")) + list(folder.glob("*.jpeg"))

    print(folder_name)
    print("  PNG :", len(pngs))
    print("  JSON:", len(jsons))
    print("  JPG/JPEG:", len(jpgs))

usable_png
  PNG : 862
  JSON: 0
  JPG/JPEG: 0
excluded_png
  PNG : 102
  JSON: 0
  JPG/JPEG: 0
